### Сравнение наборов признаков (Feature Selection) для регрессии

Эксперимент: обучить **LinearRegression** на **6 наборах признаков** и сравнить метрики.

**Наборы признаков:**

| Набор | Описание |
|-------|----------|
| A | Базовый: `op_code`, `month`, `year`, `fact_predict`, `workDayMonth_2`, `calculate_minute` |
| B | A + **все** `calc_*` из `calculate_parameter` |
| C | A + **whitelist** `calc_*`: `employeesFact`, `crossFunction`, `overtime`, `resistModel10`, `workDayMonth_2` |
| A+lags | A + `lag_1`, `lag_12`, `rolling_mean_3` |
| B+lags | B + `lag_1`, `lag_12`, `rolling_mean_3` |
| C+lags | C + `lag_1`, `lag_12`, `rolling_mean_3` |

**Тест:** июль–декабрь 2025 (time-based split).

**Метрики:** Точность, Стабильность, Качество (как в `build_metrics_dashboard_simple.py`).

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression

%matplotlib inline
import warnings
warnings.filterwarnings("ignore", "is_categorical_dtype")
warnings.filterwarnings("ignore", "use_inf_as_na")

#### Конфигурация

In [ ]:
RESIST_ID = 124

RAW_DIR = Path("../data/raw")
RAW_FILES = [
    RAW_DIR / "data-2022.csv",
    RAW_DIR / "data-2023.csv",
    RAW_DIR / "data-2024.csv",
    RAW_DIR / "data-2025.csv",
]
TEST_YEAR = 2025
TEST_MONTHS = [7, 8, 9, 10, 11, 12]

CALC_WHITELIST = [
    "calc_employeesFact",
    "calc_crossFunction",
    "calc_overtime",
    "calc_resistModel10",
    "calc_workDayMonth_2",
]

LAG_FEATURES = ["lag_1", "lag_12", "rolling_mean_3"]

#### Загрузка данных и подготовка (с calc_flat)

In [ ]:
EXCLUDED_RESIST_IDS = [
    25, 26, 27, 29, 54, 57, 61, 63, 66, 68, 80, 81, 82, 83, 100, 101, 102, 103,
    104, 105, 106, 109, 112, 113, 115, 116, 119, 120, 121, 122, 123, 125, 126,
    129, 131, 132, 134, 135, 136, 137, 138, 140, 141, 143, 144, 146, 147, 148,
    149, 150, 151, 152, 153, 154, 156, 157, 158, 162, 165, 169, 170, 171, 172,
    173, 174, 175, 176, 177, 178, 179, 181, 196, 139,
]
PREDICT_COMMENT = "first_predict_value_record"


def to_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin(["true", "t", "1"])


def to_bool_value(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return None
    return str(value).strip().lower() in ["true", "t", "1"]


def parse_calculate_param(calc_json: str):
    if pd.isna(calc_json):
        return {}
    try:
        return json.loads(calc_json)
    except Exception:
        return {}


def flatten_calc_to_dict(calc_obj: dict, prefix: str = "calc_") -> dict:
    out = {}
    for k, v in (calc_obj or {}).items():
        col = f"{prefix}{k}"
        if isinstance(v, dict):
            if "value" in v:
                out[col] = v.get("value")
            else:
                out[col] = json.dumps(v, ensure_ascii=False) if v else ""
        else:
            out[col] = v
    return out


dfs = []
for p in RAW_FILES:
    if p.exists():
        dfs.append(pd.read_csv(p, dtype=str))
if not dfs:
    raise FileNotFoundError(f"\u0424\u0430\u0439\u043b\u044b \u0434\u0430\u043d\u043d\u044b\u0445 \u043d\u0435 \u043d\u0430\u0439\u0434\u0435\u043d\u044b \u0432 {RAW_DIR}")

df_all = pd.concat(dfs, ignore_index=True)
df_all["period"] = pd.to_datetime(df_all["period"], errors="coerce")
df_all["resist_id"] = pd.to_numeric(df_all["resist_id"], errors="coerce").astype("Int64")

bool_cols = ["send", "validate_one", "validate_two", "cancel_approve", "forecast"]
for col in bool_cols:
    if col in df_all.columns:
        df_all[col] = to_bool(df_all[col])
df_all["validate_count"] = pd.to_numeric(df_all["validate_count"], errors="coerce").fillna(0).astype(int)
if "comment" in df_all.columns:
    df_all["comment"] = df_all["comment"].fillna("")
    mask_predict_record = df_all["comment"] == PREDICT_COMMENT
else:
    mask_predict_record = pd.Series(False, index=df_all.index)

mask_ok = (
    ~df_all["cancel_approve"]
    & ~mask_predict_record
    & ~df_all["resist_id"].isin(EXCLUDED_RESIST_IDS)
)
df_ok = df_all[mask_ok].copy()

records = []
for _, row in df_ok.iterrows():
    calc_obj = parse_calculate_param(row.get("calculate_parameter"))
    calc_flat = flatten_calc_to_dict(calc_obj)
    op_raw = row.get("operation_parameter")
    if pd.isna(op_raw):
        continue
    try:
        op_data = json.loads(op_raw)
    except Exception:
        continue
    period = row.get("period")
    if pd.isna(period):
        continue
    for op_id, payload in op_data.items():
        if not isinstance(payload, dict):
            continue
        if to_bool_value(payload.get("operation_active", payload.get("active"))) is False:
            continue
        base_record = {
            "resist_id": row.get("resist_id"),
            "month": int(period.month),
            "year": int(period.year),
            "operation_id": op_id,
            "fact_value": payload.get("fact_value"),
            "fact_predict": payload.get("fact_predict"),
            "workDayMonth_2": calc_flat.get("calc_workDayMonth_2"),
            "calculate_minute": payload.get("calculate_minute"),
        }
        records.append({**base_record, **calc_flat})

df_model39 = pd.DataFrame(records)
df_model39["fact_value"] = pd.to_numeric(df_model39["fact_value"], errors="coerce")
df_model39["fact_predict"] = pd.to_numeric(df_model39["fact_predict"], errors="coerce")
df_model39["workDayMonth_2"] = pd.to_numeric(df_model39["workDayMonth_2"], errors="coerce")
df_model39["calculate_minute"] = pd.to_numeric(df_model39["calculate_minute"], errors="coerce")

calc_cols = [c for c in df_model39.columns if c.startswith("calc_")]
for col in calc_cols:
    df_model39[col] = pd.to_numeric(df_model39[col], errors="coerce")
print(f"calc_* \u043a\u043e\u043b\u043e\u043d\u043a\u0438 ({len(calc_cols)}):", calc_cols)

df_model39["date"] = pd.to_datetime(
    dict(year=df_model39["year"], month=df_model39["month"], day=1), errors="coerce"
).dt.date
df_model39 = df_model39.dropna(subset=["fact_value"])
df_model39 = df_model39.sort_values(["operation_id", "year", "month"]).reset_index(drop=True)

df = df_model39[df_model39["resist_id"] == RESIST_ID].copy()
if df.empty:
    raise ValueError(
        f"\u041d\u0435\u0442 \u0434\u0430\u043d\u043d\u044b\u0445 \u0434\u043b\u044f resist_id={RESIST_ID}. "
        f"\u0414\u043e\u0441\u0442\u0443\u043f\u043d\u044b\u0435: {sorted(df_model39['resist_id'].dropna().unique()[:10].tolist())}..."
    )

df["op_code"] = df["operation_id"].astype("category").cat.codes
print(f"resist_id={RESIST_ID}: {df['operation_id'].nunique()} \u043e\u043f\u0435\u0440\u0430\u0446\u0438\u0439, {len(df)} \u0441\u0442\u0440\u043e\u043a")
print(f"calc_* \u043a\u043e\u043b\u043e\u043d\u043a\u0438 \u0432 df: {[c for c in df.columns if c.startswith('calc_')]}")

#### Feature Engineering: lag_1, lag_12, rolling_mean_3

In [ ]:
group = df.groupby("operation_id")
df["lag_1"] = group["fact_value"].shift(1)
df["lag_12"] = group["fact_value"].shift(12)
df["rolling_mean_3"] = group["fact_value"].transform(
    lambda s: s.shift(1).rolling(window=3, min_periods=1).mean()
)

num_cols = df.select_dtypes(include=["number"]).columns
df[num_cols] = df[num_cols].fillna(0)
df = df.infer_objects(copy=False)

print(f"\u041b\u0430\u0433\u043e\u0432\u044b\u0435 \u043f\u0440\u0438\u0437\u043d\u0430\u043a\u0438 \u0434\u043e\u0431\u0430\u0432\u043b\u0435\u043d\u044b: {LAG_FEATURES}")
df[["operation_id", "year", "month", "fact_value", "lag_1", "lag_12", "rolling_mean_3"]].head(15)

#### Проверка пропусков

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print("\u041f\u0440\u043e\u043f\u0443\u0441\u043a\u0438:")
    print(missing)
    df = df.fillna(0)
else:
    print("dataset \u043d\u0435 \u0441\u043e\u0434\u0435\u0440\u0436\u0438\u0442 \u043f\u0443\u0441\u0442\u044b\u0445 \u0437\u043d\u0430\u0447\u0435\u043d\u0438\u0439")

#### Определение 6 наборов признаков

In [ ]:
BASE_FEATURES = ["op_code", "month", "year", "fact_predict", "workDayMonth_2", "calculate_minute"]

available_calc_cols = [c for c in df.columns if c.startswith("calc_") and c != "calc_workDayMonth_2"]
available_whitelist = [c for c in CALC_WHITELIST if c in df.columns]

FEATURE_SETS = {
    "A": BASE_FEATURES,
    "B": BASE_FEATURES + available_calc_cols,
    "C": BASE_FEATURES + [c for c in available_whitelist if c not in BASE_FEATURES],
    "A+lags": BASE_FEATURES + LAG_FEATURES,
    "B+lags": BASE_FEATURES + available_calc_cols + LAG_FEATURES,
    "C+lags": BASE_FEATURES + [c for c in available_whitelist if c not in BASE_FEATURES] + LAG_FEATURES,
}

for name, cols in FEATURE_SETS.items():
    print(f"{name:8s} ({len(cols):2d} \u043f\u0440\u0438\u0437\u043d.): {cols}")

#### Обучение и оценка по 6 наборам

In [ ]:
def calc_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[float, float, float]:
    """\u0422\u043e\u0447\u043d\u043e\u0441\u0442\u044c, \u0421\u0442\u0430\u0431\u0438\u043b\u044c\u043d\u043e\u0441\u0442\u044c, \u041a\u0430\u0447\u0435\u0441\u0442\u0432\u043e."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 0)
    if len(y_true) == 0:
        return 0.0, 0.0, 0.0
    acc = np.empty_like(y_true)
    mask_zero = y_true == 0
    acc[mask_zero] = np.where(y_pred[mask_zero] == 0, 100.0, 0.0)
    mask_ok = ~mask_zero
    if np.any(mask_ok):
        acc[mask_ok] = np.clip(
            100.0 - np.abs(y_true[mask_ok] - y_pred[mask_ok]) / np.abs(y_true[mask_ok]) * 100.0,
            0, 100,
        )
    tochnost = round(float(np.mean(acc)), 2)
    stab = 0.0
    if tochnost > 0:
        cv = np.std(acc, ddof=0) / tochnost
        stab = round(max(0.0, (1 - cv)) * 100, 2)
    kachestvo = 0.0
    if tochnost + stab > 0:
        kachestvo = round(2 * tochnost * stab / (tochnost + stab), 2)
    return tochnost, stab, kachestvo


mask_test = (df["year"] == TEST_YEAR) & (df["month"].isin(TEST_MONTHS))
mask_train = ~mask_test

y_all = df["fact_value"]

results = []
predictions = {}

for set_name, feature_cols in FEATURE_SETS.items():
    X_all = df[feature_cols]
    X_train = X_all.loc[mask_train]
    X_test = X_all.loc[mask_test]
    y_train = y_all.loc[mask_train]
    y_test = y_all.loc[mask_test]

    if len(X_train) == 0 or len(X_test) == 0:
        print(f"{set_name}: \u043f\u0440\u043e\u043f\u0443\u0441\u043a (\u043d\u0435\u0442 \u0434\u0430\u043d\u043d\u044b\u0445 train/test)")
        continue

    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = np.maximum(model.predict(X_test), 0)

    tochnost, stab, kachestvo = calc_metrics(y_test.values, y_pred)
    t_fp, s_fp, q_fp = calc_metrics(y_test.values, df.loc[mask_test, "fact_predict"].values)

    results.append({
        "\u041d\u0430\u0431\u043e\u0440": set_name,
        "\u041f\u0440\u0438\u0437\u043d\u0430\u043a\u043e\u0432": len(feature_cols),
        "\u0422\u043e\u0447\u043d\u043e\u0441\u0442\u044c, %": tochnost,
        "\u0421\u0442\u0430\u0431\u0438\u043b\u044c\u043d\u043e\u0441\u0442\u044c, %": stab,
        "\u041a\u0430\u0447\u0435\u0441\u0442\u0432\u043e, %": kachestvo,
        "\u0422\u043e\u0447\u043d. fact_predict, %": t_fp,
        "\u0421\u0442\u0430\u0431. fact_predict, %": s_fp,
        "\u041a\u0430\u0447. fact_predict, %": q_fp,
    })
    predictions[set_name] = y_pred

    print(f"{set_name:8s} | \u0422\u043e\u0447\u043d\u043e\u0441\u0442\u044c={tochnost:6.2f}% | \u0421\u0442\u0430\u0431\u0438\u043b\u044c\u043d\u043e\u0441\u0442\u044c={stab:6.2f}% | \u041a\u0430\u0447\u0435\u0441\u0442\u0432\u043e={kachestvo:6.2f}%")

#### Сводная таблица результатов

In [ ]:
df_results = pd.DataFrame(results)
df_results = df_results.sort_values("\u041a\u0430\u0447\u0435\u0441\u0442\u0432\u043e, %", ascending=False).reset_index(drop=True)
df_results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics_to_plot = ["\u0422\u043e\u0447\u043d\u043e\u0441\u0442\u044c, %", "\u0421\u0442\u0430\u0431\u0438\u043b\u044c\u043d\u043e\u0441\u0442\u044c, %", "\u041a\u0430\u0447\u0435\u0441\u0442\u0432\u043e, %"]
for ax, metric in zip(axes, metrics_to_plot):
    df_plot = df_results.sort_values(metric, ascending=True)
    ax.barh(df_plot["\u041d\u0430\u0431\u043e\u0440"], df_plot[metric])
    ax.set_xlabel(metric)
    ax.set_title(metric)
plt.suptitle(f"\u0421\u0440\u0430\u0432\u043d\u0435\u043d\u0438\u0435 \u043d\u0430\u0431\u043e\u0440\u043e\u0432 \u043f\u0440\u0438\u0437\u043d\u0430\u043a\u043e\u0432 (resist_id={RESIST_ID})", fontsize=14)
plt.tight_layout()
plt.show()

#### Детали по операциям (лучший набор)

In [ ]:
best_set = df_results.iloc[0]["\u041d\u0430\u0431\u043e\u0440"]
print(f"\u041b\u0443\u0447\u0448\u0438\u0439 \u043d\u0430\u0431\u043e\u0440 \u043f\u043e \u041a\u0430\u0447\u0435\u0441\u0442\u0432\u0443: {best_set}")

best_pred = predictions[best_set]
df_test = df.loc[mask_test].copy()
df_test["predict_linear"] = best_pred

rows_op = []
for (rid, op_id), grp in df_test.groupby(["resist_id", "operation_id"]):
    t1, s1, q1 = calc_metrics(grp["fact_value"].values, grp["fact_predict"].values)
    t2, s2, q2 = calc_metrics(grp["fact_value"].values, grp["predict_linear"].values)
    rows_op.append({
        "resist_id": rid,
        "operation_id": op_id,
        "\u0422\u043e\u0447\u043d\u043e\u0441\u0442\u044c (fact_predict)": t1,
        "\u0421\u0442\u0430\u0431\u0438\u043b\u044c\u043d\u043e\u0441\u0442\u044c (fact_predict)": s1,
        "\u041a\u0430\u0447\u0435\u0441\u0442\u0432\u043e (fact_predict)": q1,
        f"\u0422\u043e\u0447\u043d\u043e\u0441\u0442\u044c ({best_set})": t2,
        f"\u0421\u0442\u0430\u0431\u0438\u043b\u044c\u043d\u043e\u0441\u0442\u044c ({best_set})": s2,
        f"\u041a\u0430\u0447\u0435\u0441\u0442\u0432\u043e ({best_set})": q2,
    })

df_metrics_op = pd.DataFrame(rows_op)
print(f"\u041e\u043f\u0435\u0440\u0430\u0446\u0438\u0439 \u0432 \u0442\u0435\u0441\u0442\u0435: {len(df_metrics_op)}")
df_metrics_op.head(10)

#### Экспорт в Excel

In [ ]:
df_detail = df_test[["resist_id", "operation_id", "year", "month",
                      "fact_value", "fact_predict", "predict_linear"]].copy()
df_detail = df_detail.sort_values(["resist_id", "operation_id", "year", "month"]).reset_index(drop=True)
df_detail = df_detail.rename(columns={
    "fact_value": "fact_value (\u0444\u0430\u043a\u0442)",
    "fact_predict": "fact_predict (\u043a\u043e\u043b-\u0432\u043e)",
    "predict_linear": f"\u043f\u0440\u043e\u0433\u043d\u043e\u0437 {best_set} (\u043a\u043e\u043b-\u0432\u043e)",
})

out_path = Path("../data/processed/regression_feature_selection.xlsx")
out_path.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    df_results.to_excel(w, sheet_name="\u0421\u0432\u043e\u0434\u043a\u0430 \u043d\u0430\u0431\u043e\u0440\u043e\u0432", index=False)
    df_detail.to_excel(w, sheet_name="\u0414\u0430\u043d\u043d\u044b\u0435 \u0442\u0435\u0441\u0442\u0430", index=False)
    df_metrics_op.to_excel(w, sheet_name="\u041f\u043e \u043e\u043f\u0435\u0440\u0430\u0446\u0438\u044f\u043c", index=False)

print(f"Excel \u0441\u043e\u0445\u0440\u0430\u043d\u0451\u043d: {out_path.resolve()}")

#### Выводы

* Проведено сравнение **6 наборов признаков**: A (базовый), B (все calc_*), C (whitelist calc_*), и каждый с лаговыми признаками (lag_1, lag_12, rolling_mean_3).
* **Лучший набор** определяется по метрике **Качество** (гармоническое среднее Точности и Стабильности).
* Результаты экспортированы в `regression_feature_selection.xlsx` с тремя листами:
  * **Сводка наборов** — Точность, Стабильность, Качество по каждому набору.
  * **Данные теста** — факт, fact_predict и прогноз лучшего набора.
  * **По операциям** — метрики по каждой операции для лучшего набора vs fact_predict.